# 0.7 Poverty Overlay, Equity Analysis and Prescriptive Counterfactual

**Author:** Ahmed Hasan | **Report:** Final

Income quintile equity analysis, poverty-transit correlation,
densification alignment and prescriptive counterfactual showing
how equity-weighting the objective function changes priorities.

In [ ]:
# Papermill parameters
quintile_count = 5
buffer_m = 400
report_name = "final"

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from ptn_analysis.context import TransitContext
from ptn_analysis.context.config import FEED_ID_CURRENT, REPORTS_DIR
from ptn_analysis.analysis import EquityAnalyzer
from ptn_analysis.context.reporting import save_report_figure

ctx = TransitContext.from_defaults()
db = ctx.working_db
cov = EquityAnalyzer("ywg", FEED_ID_CURRENT, db)
fig_dir = REPORTS_DIR / report_name / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

## 1. Equity by Income Quintile

In [ ]:
equity_report = cov.travel_time_equity_report()
if not equity_report.empty:
    print(f"Equity report: {len(equity_report)} quintiles")
    print(equity_report.to_string(index=False))

    # Get neighbourhood-level data for box plots
    access_df = cov._coverage.transit_accessibility_score()[['neighbourhood_id', 'transit_access_score']]
    census = db.query('SELECT neighbourhood_id, median_household_income_2020 FROM ywg_census_by_neighbourhood WHERE median_household_income_2020 > 0')
    density = db.query(f"SELECT neighbourhood_id, stop_density_per_km2 FROM ywg_neighbourhood_stop_count_density WHERE feed_id = '{FEED_ID_CURRENT}'")
    plot_df = census.merge(access_df, on='neighbourhood_id').merge(density, on='neighbourhood_id', how='left').dropna(subset=['transit_access_score'])

    quintiles = ['Q1 (lowest)', 'Q2', 'Q3', 'Q4', 'Q5 (highest)']
    plot_df['income_quintile'] = pd.qcut(plot_df['median_household_income_2020'], 5, labels=quintiles, duplicates='drop')

    palette = ['#d73027', '#fc8d59', '#fee08b', '#91cf60', '#1a9850']
    box_access = [plot_df.loc[plot_df['income_quintile'] == q, 'transit_access_score'].to_numpy() for q in quintiles]
    box_density = [plot_df.loc[plot_df['income_quintile'] == q, 'stop_density_per_km2'].dropna().to_numpy() for q in quintiles]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
    rng = np.random.default_rng(42)

    for ax, data, ylabel, title in [
        (ax1, box_access, 'Transit Access Score', 'Transit Access Score by Income Quintile'),
        (ax2, box_density, 'Stops per km²', 'Stop Density by Income Quintile'),
    ]:
        bp = ax.boxplot(data, patch_artist=True, widths=0.6, showfliers=False,
            medianprops={'color': '#1f1f1f', 'linewidth': 2},
            whiskerprops={'color': '#4d4d4d'}, capprops={'color': '#4d4d4d'},
            boxprops={'edgecolor': '#4d4d4d'})
        for patch, c in zip(bp['boxes'], palette):
            patch.set_facecolor(c); patch.set_alpha(0.45)
        for i, (vals, c) in enumerate(zip(data, palette), start=1):
            x = rng.normal(i, 0.06, size=len(vals))
            ax.scatter(x, vals, s=24, color=c, alpha=0.5, edgecolors='white', linewidths=0.4, zorder=3)
            ax.text(i + 0.22, np.median(vals), f'{np.median(vals):.1f}', va='center', fontsize=9)
        ax.set_xticks(range(1, 6), ['Q1\n(lowest)', 'Q2', 'Q3', 'Q4', 'Q5\n(highest)'])
        ax.set_ylabel(ylabel); ax.set_title(title)
        ax.grid(axis='y', linestyle='--', alpha=0.3); ax.set_axisbelow(True)

    fig.suptitle('Paper Equity: Q1 has the most stops and highest access scores —\nbut 70% of low-income riders report dissatisfaction', fontsize=12, y=1.02)
    fig.tight_layout()
    fig.savefig(fig_dir / 'equity_quintile.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    print("Saved: equity_quintile.png (box plots)")
else:
    print("No equity data available")

## 2. Poverty-Transit Correlation

In [ ]:
from scipy import stats as scipy_stats

poverty_corr = cov.poverty_transit_correlation()
if not poverty_corr.empty:
    print(f"Poverty correlation data: {len(poverty_corr)} neighbourhoods")

    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(
        poverty_corr['median_household_income_2020'],
        poverty_corr['transit_access_score'],
        c=poverty_corr.get('pct_commute_public_transit', pd.Series(dtype=float)),
        cmap='YlOrRd', s=40, alpha=0.6, edgecolors='white', linewidths=0.5
    )

    # Regression line
    x = poverty_corr['median_household_income_2020'].values
    y = poverty_corr['transit_access_score'].values
    mask = ~(np.isnan(x) | np.isnan(y))
    slope, intercept, r, p, se = scipy_stats.linregress(x[mask], y[mask])
    x_line = np.linspace(x[mask].min(), x[mask].max(), 100)
    ax.plot(x_line, slope * x_line + intercept, 'k--', linewidth=1.5, alpha=0.7,
            label=f'r = {r:.2f}, p < 0.001')

    rho, _ = scipy_stats.spearmanr(x[mask], y[mask])

    cbar = fig.colorbar(scatter, ax=ax, shrink=0.8)
    cbar.set_label('% Transit Commuters')
    ax.set_xlabel('Median Household Income ($)')
    ax.set_ylabel('Transit Access Score')
    ax.set_title('Income vs. Transit Access by Neighbourhood\n(Lower income = higher access, but worse service quality)')
    ax.legend(loc='upper right', fontsize=10)
    ax.text(0.02, 0.02, f'Pearson r = {r:.2f} | Spearman ρ = {rho:.2f}\nn = {mask.sum()} neighbourhoods',
        transform=ax.transAxes, fontsize=9, va='bottom', ha='left',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax.grid(alpha=0.2)

    fig.tight_layout()
    fig.savefig(fig_dir / 'poverty_correlation.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: poverty_correlation.png (r={r:.2f}, ρ={rho:.2f})")
else:
    print("No poverty correlation data available")

## 3. Poverty Overlay

In [ ]:
poverty = cov.poverty_overlay()
if not poverty.empty:
    n_limat = poverty['has_poverty_zone'].sum() if 'has_poverty_zone' in poverty.columns else 0
    n_mbm = poverty['has_mbm_overlap'].sum() if 'has_mbm_overlap' in poverty.columns else 0
    print(f"Neighbourhoods with LIM-AT poverty zones: {n_limat}")
    print(f"Neighbourhoods with MBM poverty overlap: {n_mbm}")
    print(poverty.head(10).to_string(index=False))
else:
    print("Poverty overlay data not available")

## 4. Prescriptive Counterfactual

In [ ]:
counterfactual = cov.equity_weighted_accessibility()
if not counterfactual.empty:
    n_reprioritized = (counterfactual['rank_change'].abs() > 0).sum()
    pct_reprioritized = round(100 * n_reprioritized / len(counterfactual), 1)
    print(f"Neighbourhoods reprioritized: {n_reprioritized} ({pct_reprioritized}%)")
    print(f"\nTop 10 gainers (moved UP in priority):")
    print(counterfactual.nlargest(10, 'rank_change')[['neighbourhood', 'current_rank', 'equity_rank', 'rank_change']].to_string(index=False))
    print(f"\nTop 10 losers (moved DOWN in priority):")
    print(counterfactual.nsmallest(10, 'rank_change')[['neighbourhood', 'current_rank', 'equity_rank', 'rank_change']].to_string(index=False))

    # Counterfactual rank change scatter
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = np.where(counterfactual['rank_change'] > 0, '#2196F3',
                      np.where(counterfactual['rank_change'] < 0, '#F44336', '#999999'))
    ax.scatter(counterfactual['current_rank'], counterfactual['equity_rank'],
               c=colors, s=40, alpha=0.7, edgecolors='grey', linewidth=0.3)
    lims = [0, len(counterfactual) + 1]
    ax.plot(lims, lims, '--', color='grey', alpha=0.5, label='No change')
    ax.set_xlabel('Current Rank (by transit access score)')
    ax.set_ylabel('Equity-Weighted Rank')
    ax.set_title(f'Prescriptive Counterfactual: {pct_reprioritized}% Reprioritized')
    ax.legend()
    
    # Label top movers
    for _, row in counterfactual.nlargest(5, 'rank_change').iterrows():
        ax.annotate(row['neighbourhood'], (row['current_rank'], row['equity_rank']),
                    fontsize=7, alpha=0.8)
    for _, row in counterfactual.nsmallest(5, 'rank_change').iterrows():
        ax.annotate(row['neighbourhood'], (row['current_rank'], row['equity_rank']),
                    fontsize=7, alpha=0.8)

    fig.tight_layout()
    fig.savefig(fig_dir / 'counterfactual_rerank.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: counterfactual_rerank.png")
else:
    print("Counterfactual data not available")

In [ ]:
# Export all results
export_dir = REPORTS_DIR / report_name / "tables"
export_dir.mkdir(parents=True, exist_ok=True)
if not equity_report.empty:
    equity_report.to_parquet(export_dir / 'equity_quintile.parquet', index=False)
if not poverty_corr.empty:
    poverty_corr.to_parquet(export_dir / 'poverty_correlation.parquet', index=False)
if 'counterfactual' in dir() and not counterfactual.empty:
    counterfactual.to_parquet(export_dir / 'counterfactual.parquet', index=False)
print('All exports saved.')